# 02 - BM25 Retrieval

**No API key. Uses rank-bm25 library.**

BM25 (Best Match 25) is an improved retrieval algorithm that addresses two
weaknesses in plain TF-IDF:

1. **Term saturation**: A word appearing 20 times should not score 20x higher
   than a word appearing once. BM25 caps the contribution of repeated words.
2. **Document length normalization**: A short chunk that mentions a word once
   should score higher than a long chunk that mentions it once buried among
   hundreds of other words.

BM25 is the default retrieval algorithm used by search engines like
Elasticsearch and Apache Lucene.

### What you will learn

| Step | Concept |
|------|---------|
| 1 | Understand BM25 parameters k1 and b |
| 2 | Build a BM25 index with rank_bm25 |
| 3 | Retrieve top-k results for any query |
| 4 | Compare BM25 vs TF-IDF on the same queries |

> No internet connection or API key required.

## 1. Install and Import

In [ ]:
# !pip install rank-bm25

In [ ]:
import re
import math
from pathlib import Path
from collections import Counter
from rank_bm25 import BM25Okapi

print("rank_bm25 imported successfully")

## 2. Load and Chunk the Documents

In [ ]:
DATA_DIR = Path("../data")

STOP_WORDS = {
    "a", "an", "the", "is", "it", "in", "on", "at", "to", "of",
    "and", "or", "but", "for", "with", "by", "from", "as", "was",
    "are", "were", "be", "been", "has", "have", "had", "this", "that",
    "these", "those", "its", "their", "they", "also", "which", "into",
    "more", "than", "about", "such", "can", "not", "over", "up", "out"
}

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]


all_chunks = []
for filepath in sorted(DATA_DIR.glob("*.txt")):
    text = filepath.read_text()
    for para in text.split("\n\n"):
        cleaned = para.strip()
        if len(cleaned) >= 50:
            all_chunks.append({"source": filepath.name, "text": cleaned})

print(f"Total chunks: {len(all_chunks)}")
print(f"Sample chunk sources: {list(set(c['source'] for c in all_chunks))}")

## 3. Understanding BM25 Parameters

BM25 has two tuning parameters:

- **k1** (default 1.5): Controls term saturation. Higher k1 means more reward
  for repeated terms. Typical range: 1.2 to 2.0.
- **b** (default 0.75): Controls document length normalization. b=0 means no
  length normalization; b=1 means full normalization. Typical value: 0.75.

The formula is:

```
BM25(word, chunk) = IDF(word) * TF(word, chunk) * (k1 + 1)
                   / (TF(word, chunk) + k1 * (1 - b + b * len(chunk)/avg_len))
```

You rarely need to change these defaults.

## 4. Build the BM25 Index

In [ ]:
# Tokenize every chunk once and store
tokenized_chunks = [tokenize(chunk["text"]) for chunk in all_chunks]

# BM25Okapi expects a list of token lists
bm25_index = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")
print(f"Average document length: {bm25_index.avgdl:.1f} tokens")

## 5. Retrieve with BM25

In [ ]:
def bm25_retrieve(query, top_k=3):
    """Return the top-k chunks most relevant to the query."""    
    query_tokens = tokenize(query)
    scores       = bm25_index.get_scores(query_tokens)

    # Pair each score with its chunk and sort descending
    ranked = sorted(
        zip(scores, all_chunks),
        key=lambda x: x[0],
        reverse=True
    )
    return ranked[:top_k]


def ask_bm25(query, top_k=3):
    print(f"Query: {query}")
    print("-" * 55)
    for rank, (score, chunk) in enumerate(bm25_retrieve(query, top_k), 1):
        preview = chunk["text"][:160].replace("\n", " ")
        print(f"[{rank}] Score: {score:.4f}  Source: {chunk['source']}")
        print(f"    {preview}...")
        print()

ask_bm25("Who built the Great Pyramid of Giza?")

In [ ]:
ask_bm25("What causes sea levels to rise?")

In [ ]:
ask_bm25("How do neural networks learn?")

## 6. Side-by-Side: BM25 vs TF-IDF

Let us reproduce the TF-IDF scorer from Notebook 01 here and compare results
on the same query.

In [ ]:
# --- Reproduce TF-IDF from notebook 01 ---

def tfidf_build_idf(chunks, tokenize_fn):
    N = len(chunks)
    doc_freq = Counter()
    for chunk in chunks:
        doc_freq.update(set(tokenize_fn(chunk["text"])))
    return {w: math.log(N / df) for w, df in doc_freq.items()}

def tfidf_score_chunk(chunk_tokens, query_tokens, idf):
    if not chunk_tokens:
        return 0.0
    counts = Counter(chunk_tokens)
    total  = len(chunk_tokens)
    return sum(
        (counts[w] / total) * idf.get(w, 0)
        for w in query_tokens
        if w in counts
    )

def tfidf_retrieve(query, chunks, idf, tokenize_fn, top_k=3):
    q_toks = tokenize_fn(query)
    ranked = sorted(
        chunks,
        key=lambda c: tfidf_score_chunk(tokenize_fn(c["text"]), q_toks, idf),
        reverse=True
    )
    return ranked[:top_k]

idf_table = tfidf_build_idf(all_chunks, tokenize)

# --- Compare ---
QUERY = "What is the Amazon rainforest?"

print("=" * 55)
print(f"Query: {QUERY}")
print("=" * 55)

print("\n[TF-IDF results]")
for rank, chunk in enumerate(tfidf_retrieve(QUERY, all_chunks, idf_table, tokenize), 1):
    print(f"  [{rank}] {chunk['source']:30s}  {chunk['text'][:80]}...")

print("\n[BM25 results]")
for rank, (score, chunk) in enumerate(bm25_retrieve(QUERY), 1):
    print(f"  [{rank}] {chunk['source']:30s}  {chunk['text'][:80]}...")

## 7. When BM25 Wins Over TF-IDF

BM25 tends to produce better rankings when:

- Chunks have very different lengths (long chunks can dominate TF-IDF unfairly)
- A key term repeats many times in a single chunk (TF-IDF over-rewards repetition)
- You are searching across a large collection of documents

TF-IDF is simpler and easier to inspect, making it a good starting point for
understanding retrieval. BM25 is the production-grade choice.

### Summary

| Feature | TF-IDF | BM25 |
|---------|--------|------|
| Term saturation | No cap | Soft cap via k1 |
| Length normalization | No | Yes, via b |
| Library needed | No | rank-bm25 |
| Typical use | Learning, small datasets | Production search systems |

**Next notebook:** use BM25 for retrieval and a sliding window chunker
to improve coverage of long documents.